## RAG Day 3 (Open Source edition)

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

This is `week5/day3.ipynb` with the model swapped from `gpt-4.1-nano` to **Llama 3.2** running locally via [Ollama](https://ollama.com). No API key, no API cost - everything runs on your own machine.

**Prerequisites:**
- Ollama installed and running, with the model pulled: `ollama pull llama3.2`
- The VectorStore built in `day2_ollama.ipynb` (HuggingFace `all-MiniLM-L6-v2` embeddings, stored in `vector_db`)

So both halves of the pipeline - the embeddings *and* the LLM - are open source and local.

In [1]:
import requests
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
# No API key needed - Llama 3.2 runs locally via Ollama

OLLAMA_PORT = 11434
OLLAMA_HOST = f"http://localhost:{OLLAMA_PORT}"
MODEL = "llama3.2"
DB_NAME = "vector_db"
load_dotenv(override=True)

False

In [3]:
# Sanity check: is Ollama up, and is our model pulled?

try:
    models = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()["models"]
except Exception as e:
    raise SystemExit(f"Could not reach Ollama at {OLLAMA_HOST} - is it running? ({e})")

names = [m["name"] for m in models]
print(f"Ollama is up at {OLLAMA_HOST}. Models available: {names}")

if not any(n == MODEL or n.startswith(f"{MODEL}:") for n in names):
    print(f"\nWARNING: '{MODEL}' not found. Run:  ollama pull {MODEL}")

Ollama is up at http://localhost:11434. Models available: ['llama3.2:latest', 'gpt-oss:latest', 'phi3:latest', 'gemma3:270m']


### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

Same embedding model that built the store in `day2_ollama.ipynb` - the embeddings must match, or retrieval returns nonsense.

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
print(f"Vectorstore loaded with {vectorstore._collection.count()} documents")

Vectorstore loaded with 413 documents


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

`ChatOllama` is the drop-in replacement for `ChatOpenAI` here - same `invoke()` interface, same message objects, just pointed at your local Ollama server.

In [5]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model=MODEL, temperature=0, base_url=OLLAMA_HOST)

### These LangChain objects implement the method `invoke()`

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='c96323f0-5f98-48d3-8cf6-250d62ab16ae', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery, so it\'s possible that you\'re referring to one of the following:\n\n1. Avery Brooks: An American actor known for his roles in TV shows such as "Star Trek: Deep Space Nine" and "Designated Survivor".\n2. Avery Hunt: An American football player who played in the NFL.\n3. Avery Johnson: An American basketball coach and former player, best known for coaching the NBA\'s Dallas Mavericks.\n4. Avery Winter: A Canadian professional ice hockey player.\n\nIf you could provide more context or information about the Avery you\'re referring to, I\'d be happy to try and provide a more specific answer.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-07T09:11:17.7877154Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12561566200, 'load_duration': 3200389900, 'prompt_eval_count': 29, 'prompt_eval_duration': 328322000, 'eval_count': 133, 'eval_duration': 9008287000, 'model_name': 'llama

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [11]:
answer_question("Who is Avery Lancaster?", [])

"Avery Lancaster is the Co-Founder & Chief Executive Officer (CEO) of Insurellm. She has been instrumental in shaping the company's success and growth in the insurance technology landscape. With her innovative leadership strategies, risk management expertise, and commitment to diversity and inclusion, Avery has positioned Insurellm as a leading player in the industry.\n\nAvery has a strong background in product development and management, having worked as a Senior Product Manager at Innovate Insurance Solutions before co-founding Insurellm. Under her leadership, the company has achieved significant milestones, including market leadership, innovative product launches, and improved employee morale.\n\nAs a leader, Avery is known for her resilience, adaptability, and ability to navigate complex challenges. She has been recognized in industry publications as a leading voice in Insurance Tech innovation and has received numerous awards for her contributions to the company's growth and succe

### A note on answer quality with a small local model

Llama 3.2 is a 3B model - it is *much* smaller than `gpt-4.1-nano`, and you will notice it. Retrieval itself is unchanged (the same embeddings find the same chunks), but the model is weaker at grounding its answer in the context it was handed, so it hedges more and occasionally leans on its own priors instead of the documents.

If answers feel thin, things worth trying:
- Raise the number of retrieved chunks: `vectorstore.as_retriever(search_kwargs={"k": 25})`
- Tighten the system prompt - e.g. instruct it to answer *only* from the context
- Step up to a bigger local model (`ollama pull llama3.1:8b`) and change `MODEL`

## What could possibly come next? 😂

In [13]:
gr.ChatInterface(answer_question).launch()

c:\AI\Projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!

And this whole thing just ran on your own machine - no API key, no per-token cost, no data leaving your laptop.